# LeetCode 2946 — Matrix Similarity After Cyclic Shifts bằng Python

Link đề: https://leetcode.com/problems/matrix-similarity-after-cyclic-shifts/ · Độ khó: **Easy** · Chủ đề: Array, Math, Matrix, Simulation

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def areSimilar(self, mat: List[List[int]], k: int) -> bool:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ (đặc biệt ví dụ 2).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: dịch vòng (**cyclic shift**) các hàng chẵn sang **trái** và hàng lẻ sang **phải** đúng `k` lần, rồi hỏi ma trận có **giống hệt** ban đầu không.
2. Biết **cắt lát (slicing)** `row[k:] + row[:k]` để xoay một danh sách, và dùng **phép chia dư `%`** để "quấn vòng" chỉ số.
3. Hiểu hai nhận xét quan trọng: **`k` chỉ cần xét theo `k % n`**, và **dịch trái hay phải `k` lần cho ra ma trận giống ban đầu hay không là như nhau**.
4. Đi từ cách **mô phỏng từng bước dịch** → **xoay một lần bằng slicing** → **so sánh chỉ số `(j + k) % n` mà không xoay** → **chu kỳ `gcd(n, k)`**.
5. Tránh các lỗi kinh điển: quên `k % n` khi cắt lát, sửa luôn ma trận gốc rồi so sánh với chính nó, `return True` quá sớm trong vòng lặp, và nhầm chỉ số hàng chẵn/lẻ.

## 2. Đề bài & ví dụ

Cho ma trận số nguyên `mat` kích thước `m x n` (chỉ số từ `0`) và số nguyên `k`. Ta thực hiện `k` lần dịch vòng như sau:

- Các hàng có **chỉ số chẵn** (`0, 2, 4, ...`) được dịch vòng sang **trái**;
- Các hàng có **chỉ số lẻ** (`1, 3, 5, ...`) được dịch vòng sang **phải**.

Trả về `True` nếu sau `k` lần dịch ma trận **giống hệt** ma trận ban đầu, ngược lại trả về `False`.

Dịch vòng sang trái một lần: phần tử đầu của hàng chuyển xuống cuối, các phần tử còn lại lùi một ô. Ví dụ `[1, 2, 3] -> [2, 3, 1]`. Dịch phải một lần: phần tử cuối chuyển lên đầu, ví dụ `[1, 2, 3] -> [3, 1, 2]`.

**Ví dụ 1**

- Input: `mat = [[1,2,3],[4,5,6],[7,8,9]]`, `k = 4`
- Output: `False`
- Giải thích: mỗi hàng có 3 phần tử, dịch `4` lần cũng như dịch `1` lần. Hàng 0: `[1,2,3] -> [2,3,1]`, khác ban đầu.

**Ví dụ 2**

- Input: `mat = [[1,2,1,2],[5,5,5,5],[6,3,6,3]]`, `k = 2`
- Output: `True`
- Giải thích: dịch `2` ô thì `[1,2,1,2]` giữ nguyên (chu kỳ 2), `[5,5,5,5]` luôn giữ nguyên, `[6,3,6,3]` cũng giữ nguyên.

**Ví dụ 3**

- Input: `mat = [[2,2],[2,2]]`, `k = 3`
- Output: `True`
- Giải thích: mọi phần tử bằng nhau nên dịch bao nhiêu lần ma trận cũng không đổi.

**Ràng buộc:** `1 <= m, n <= 25`; `1 <= mat[i][j] <= 25`; `1 <= k <= 50`.

In [1]:
from typing import List
from math import gcd
from collections import deque
import random

examples = [
    ([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 4, False),
    ([[1, 2, 1, 2], [5, 5, 5, 5], [6, 3, 6, 3]], 2, True),
    ([[2, 2], [2, 2]], 3, True),
]
for mat, k, expected in examples:
    print("mat =", mat, "k =", k, "-> kỳ vọng", expected)

mat = [[1, 2, 3], [4, 5, 6], [7, 8, 9]] k = 4 -> kỳ vọng False
mat = [[1, 2, 1, 2], [5, 5, 5, 5], [6, 3, 6, 3]] k = 2 -> kỳ vọng True
mat = [[2, 2], [2, 2]] k = 3 -> kỳ vọng True


## 3. Kiến thức nền

### 3.1. Ma trận là "danh sách của các danh sách"

`mat[i]` là **hàng thứ `i`** (một danh sách), `mat[i][j]` là **phần tử ở hàng `i`, cột `j`**. Số hàng là `len(mat)`, số cột là `len(mat[0])`. Duyệt cả hàng lẫn chỉ số bằng `for i, row in enumerate(mat)`.

In [2]:
mat = [[1, 2, 3], [4, 5, 6]]
print(len(mat), len(mat[0]))              # 2 hàng, 3 cột
print(mat[1], mat[1][2])                  # [4, 5, 6] 6
for i, row in enumerate(mat):
    print("hàng", i, "=", row)

2 3
[4, 5, 6] 6
hàng 0 = [1, 2, 3]
hàng 1 = [4, 5, 6]


### 3.2. Hàng chẵn / hàng lẻ: `i % 2`

`i % 2 == 0` nghĩa là `i` **chẵn**; `i % 2 == 1` nghĩa là `i` **lẻ**. Chú ý đề đánh chỉ số từ `0`, nên hàng **đầu tiên** (chỉ số `0`) là hàng **chẵn**.

In [3]:
for i in range(5):
    print(i, "chẵn -> dịch trái" if i % 2 == 0 else "lẻ -> dịch phải")

0 chẵn -> dịch trái
1 lẻ -> dịch phải
2 chẵn -> dịch trái
3 lẻ -> dịch phải
4 chẵn -> dịch trái


### 3.3. Cắt lát (slicing) để xoay danh sách

`a[p:]` là phần từ chỉ số `p` đến hết, `a[:p]` là phần trước chỉ số `p`. Ghép `a[p:] + a[:p]` ta được danh sách **dịch vòng trái `p` ô**. Dịch phải `p` ô là `a[-p:] + a[:-p]` (với `0 < p < n`).

In [4]:
a = [1, 2, 3, 4, 5]
print(a[2:], a[:2])                       # [3, 4, 5] [1, 2]
print(a[2:] + a[:2])                      # dịch trái 2: [3, 4, 5, 1, 2]
print(a[-2:] + a[:-2])                    # dịch phải 2: [4, 5, 1, 2, 3]

[3, 4, 5] [1, 2]
[3, 4, 5, 1, 2]
[4, 5, 1, 2, 3]


### 3.4. Phép chia dư `%` để "quấn vòng"

Với `n` phần tử, dịch `n` ô thì **về chỗ cũ**. Nên chỉ cần `k % n`. Chỉ số quấn vòng: ô `j` sau khi dịch trái `k` ô nhận giá trị ban đầu của ô `(j + k) % n`.

**Cẩn thận:** cắt lát với `p >= n` **không** tự quấn vòng. `a[7:] + a[:7]` với `n = 5` là `[] + a` = `a` (chưa dịch!), trong khi dịch trái 7 ô thực ra bằng dịch trái 2 ô.

In [5]:
a = [1, 2, 3, 4, 5]
n = len(a)
print(7 % n)                                    # 2
print(a[7:] + a[:7])                            # [1, 2, 3, 4, 5]  <- SAI: không quấn vòng
print(a[7 % n:] + a[:7 % n])                    # [3, 4, 5, 1, 2]  <- đúng
print([a[(j + 7) % n] for j in range(n)])       # [3, 4, 5, 1, 2]  <- cách dùng chỉ số

2
[1, 2, 3, 4, 5]
[3, 4, 5, 1, 2]
[3, 4, 5, 1, 2]


### 3.5. `all(...)` và so sánh hai danh sách

`all(điều_kiện for ...)` trả `True` nếu **mọi** điều kiện đúng (và dừng ngay khi gặp một cái sai). Hai danh sách so sánh bằng `==` thì bằng khi cùng độ dài và từng cặp phần tử bằng nhau.

In [6]:
print(all(x > 0 for x in [3, 1, 2]))      # True
print(all(x > 1 for x in [3, 1, 2]))      # False
print([1, 2, 3] == [1, 2, 3], [1, 2, 3] == [3, 2, 1])   # True False

True
False
True False


### 3.6. Sao chép danh sách: `row[:]`, `list(row)`

`b = a` **không** tạo bản sao: `b` và `a` là **cùng một đối tượng**. Sửa `b` là sửa `a`. Muốn bản sao độc lập (tầng ngoài) dùng `a[:]` hoặc `list(a)`; với ma trận cần sao chép **từng hàng**: `[row[:] for row in mat]`.

In [7]:
a = [1, 2, 3]
b = a
b[0] = 99
print(a)                                  # [99, 2, 3]  <- a cũng bị đổi
c = [1, 2, 3]
d = c[:]
d[0] = 99
print(c, d)                               # [1, 2, 3] [99, 2, 3]

[99, 2, 3]
[1, 2, 3] [99, 2, 3]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Mỗi hàng được xử lý **độc lập**: hàng nào bị dịch xong khác ban đầu thì đáp án là `False`; nếu **mọi** hàng đều giữ nguyên thì `True`. Tính tay Ví dụ 2 (`k = 2`, `n = 4`):

| hàng `i` | hướng | hàng ban đầu | sau khi dịch `2` ô | giống ban đầu? |
|---|---|---|---|---|
| 0 | trái | `[1,2,1,2]` | `[1,2,1,2]` | có |
| 1 | phải | `[5,5,5,5]` | `[5,5,5,5]` | có |
| 2 | trái | `[6,3,6,3]` | `[6,3,6,3]` | có |

Mọi hàng đều giống nên kết quả `True`. Ví dụ 1 (`k = 4`, `n = 3`): `k % n = 1`, hàng 0 trái `1` ô là `[2,3,1]` khác `[1,2,3]` ⇒ `False`.

**Nhận xét hay:** dịch trái `k` ô rồi so với bản đầu **cho cùng kết quả** với dịch phải `k` ô. Lý do: nếu `L(row) == row` thì áp dụng dịch phải `k` ô vào hai vế được `row == R(row)`. Vậy **hướng dịch không quan trọng**, chỉ cần biết hàng có "bất biến dưới dịch `k` ô" hay không.

In [8]:
row = [1, 2, 1, 2]
n = len(row)
for k in range(1, 6):
    kk = k % n
    left = row[kk:] + row[:kk]
    right = row[n - kk:] + row[:n - kk] if kk else row[:]
    print(f"k={k}: trái={left} phải={right} | trái==gốc? {left == row} phải==gốc? {right == row}")

k=1: trái=[2, 1, 2, 1] phải=[2, 1, 2, 1] | trái==gốc? False phải==gốc? False
k=2: trái=[1, 2, 1, 2] phải=[1, 2, 1, 2] | trái==gốc? True phải==gốc? True
k=3: trái=[2, 1, 2, 1] phải=[2, 1, 2, 1] | trái==gốc? False phải==gốc? False
k=4: trái=[1, 2, 1, 2] phải=[1, 2, 1, 2] | trái==gốc? True phải==gốc? True
k=5: trái=[2, 1, 2, 1] phải=[2, 1, 2, 1] | trái==gốc? False phải==gốc? False


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng lần dịch

**Ý tưởng.** Làm đúng như đề: lặp `k` lần, mỗi lần dịch **một ô** cho từng hàng (hàng chẵn sang trái, hàng lẻ sang phải) trên một **bản sao** của ma trận, cuối cùng so sánh với ma trận gốc. Dịch trái 1 ô: `row[1:] + row[:1]`; dịch phải 1 ô: `row[-1:] + row[:-1]`.

In [9]:
def are_similar_simulate(mat: List[List[int]], k: int) -> bool:
    cur = [row[:] for row in mat]                    # bản sao độc lập, giữ nguyên mat gốc
    for _ in range(k):                               # k lần dịch
        for i in range(len(cur)):
            if i % 2 == 0:
                cur[i] = cur[i][1:] + cur[i][:1]     # trái 1 ô
            else:
                cur[i] = cur[i][-1:] + cur[i][:-1]   # phải 1 ô
    return cur == mat


for mat, k, expected in examples:
    print(mat, k, "->", are_similar_simulate(mat, k), "(kỳ vọng", expected, ")")

[[1, 2, 3], [4, 5, 6], [7, 8, 9]] 4 -> False (kỳ vọng False )
[[1, 2, 1, 2], [5, 5, 5, 5], [6, 3, 6, 3]] 2 -> True (kỳ vọng True )
[[2, 2], [2, 2]] 3 -> True (kỳ vọng True )


In [10]:
# Xem từng lần dịch của ví dụ 1 (k = 4)
mat = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
cur = [row[:] for row in mat]
for step in range(1, 5):
    for i in range(len(cur)):
        cur[i] = cur[i][1:] + cur[i][:1] if i % 2 == 0 else cur[i][-1:] + cur[i][:-1]
    print(f"sau lần {step}: {cur} | giống gốc? {cur == mat}")

sau lần 1: [[2, 3, 1], [6, 4, 5], [8, 9, 7]] | giống gốc? False
sau lần 2: [[3, 1, 2], [5, 6, 4], [9, 7, 8]] | giống gốc? False
sau lần 3: [[1, 2, 3], [4, 5, 6], [7, 8, 9]] | giống gốc? True
sau lần 4: [[2, 3, 1], [6, 4, 5], [8, 9, 7]] | giống gốc? False


**Độ phức tạp (Cách 1).** Gọi `m` hàng, `n` cột. Mỗi lần dịch tạo lại mọi hàng (`O(m·n)`), lặp `k` lần: **Time `O(k · m · n)`**; bản sao ma trận: **Space `O(m · n)`**.

**Nút thắt.** Ta dịch từng ô một, `k` lần, dù chỉ cần biết trạng thái **cuối cùng**. Ta có thể nhảy thẳng tới đó.

## 6. Ý tưởng thuật toán / bước đột phá

Hai nhận xét làm bài ngắn lại:

1. **Dịch `n` ô thì về chỗ cũ**, nên dịch `k` ô giống dịch `k % n` ô. Ta xoay **một lần** bằng slicing thay vì `k` lần.
2. **Hướng dịch không ảnh hưởng đáp án** (mục 4). Vậy với **mọi** hàng, chỉ cần kiểm tra: dịch trái `k % n` ô có ra đúng hàng ban đầu không. Thậm chí khỏi tạo hàng mới: hàng bất biến khi và chỉ khi **`row[j] == row[(j + k) % n]` với mọi `j`**.

Hệ quả: `i % 2` (chẵn/lẻ) trở nên **không cần thiết**, kết quả chỉ là `all` các hàng bất biến.

## 7. Các cách tiếp theo

### Cách 2 — Xoay một lần bằng slicing, theo đúng hướng đề

**Giải thích.** Đặt `s = k % n`. Hàng chẵn: `row[s:] + row[:s]` (trái `s`). Hàng lẻ: `row[n - s:] + row[:n - s]` (phải `s`). So sánh với `row`; nếu có hàng nào khác thì trả `False`. Ta vẫn làm theo đề nhưng chỉ xoay **một lần mỗi hàng**.

In [11]:
def are_similar_slice(mat: List[List[int]], k: int) -> bool:
    n = len(mat[0])
    s = k % n                                        # dịch s ô đủ thay cho k ô
    for i, row in enumerate(mat):
        if i % 2 == 0:
            shifted = row[s:] + row[:s]              # trái s
        else:
            shifted = row[n - s:] + row[:n - s]      # phải s (s = 0 thì row[n:] = [] + row[:n] = row)
        if shifted != row:
            return False                             # một hàng đổi là xong
    return True


for mat, k, expected in examples:
    print(mat, k, "->", are_similar_slice(mat, k), "(kỳ vọng", expected, ")")

[[1, 2, 3], [4, 5, 6], [7, 8, 9]] 4 -> False (kỳ vọng False )
[[1, 2, 1, 2], [5, 5, 5, 5], [6, 3, 6, 3]] 2 -> True (kỳ vọng True )
[[2, 2], [2, 2]] 3 -> True (kỳ vọng True )


**Độ phức tạp Cách 2:** **Time `O(m · n)`** (mỗi hàng một lần xoay + so sánh), **Space `O(n)`** cho một hàng tạm.

### Cách 3 — So sánh chỉ số `(j + k) % n`, không xoay

**Giải thích.** Sau khi dịch trái `k` ô, ô `j` chứa giá trị ban đầu của ô `(j + k) % n`. Hàng giữ nguyên khi và chỉ khi `row[j] == row[(j + k) % n]` với mọi `j`. Không cần tạo danh sách mới, và (nhờ nhận xét về hướng) không cần xét hàng chẵn/lẻ.

In [12]:
def are_similar_index(mat: List[List[int]], k: int) -> bool:
    n = len(mat[0])
    for row in mat:
        for j in range(n):
            if row[j] != row[(j + k) % n]:           # k lớn cũng không sao: % n quấn vòng
                return False
    return True


def are_similar_index_all(mat: List[List[int]], k: int) -> bool:
    n = len(mat[0])
    return all(row[j] == row[(j + k) % n] for row in mat for j in range(n))


for mat, k, expected in examples:
    print(are_similar_index(mat, k), are_similar_index_all(mat, k), "(kỳ vọng", expected, ")")

False False (kỳ vọng False )
True True (kỳ vọng True )
True True (kỳ vọng True )


**Độ phức tạp Cách 3:** **Time `O(m · n)`**, **Space `O(1)`** (không tạo danh sách mới).

### Cách 4 — Chu kỳ `gcd(n, k)`

**Giải thích.** Một hàng bất biến khi dịch `k` ô ⇔ bất biến khi dịch `g = gcd(n, k)` ô (vì các bội của `k` modulo `n` sinh ra đúng các bội của `g`). Nên chỉ cần kiểm tra `row[j] == row[(j + g) % n]`. Ý nghĩa: hàng lặp lại với **chu kỳ `g`**. Ví dụ `[1,2,1,2]` có chu kỳ `2`; `n = 4`, `k = 6` ⇒ `gcd(4, 6) = 2` nên bất biến. Cách này thú vị về mặt toán, không nhanh hơn rõ rệt ở bài này.

In [13]:
def are_similar_gcd(mat: List[List[int]], k: int) -> bool:
    n = len(mat[0])
    g = gcd(n, k)                                    # gcd(n, k) = gcd(n, k % n)
    return all(row[j] == row[(j + g) % n] for row in mat for j in range(n))


for mat, k, expected in examples:
    print(are_similar_gcd(mat, k), "(kỳ vọng", expected, ")")

# Quan sát: các giá trị k % n cho cùng kết quả bất biến với gcd(n, k)
n, row = 6, [1, 2, 3, 1, 2, 3]                       # chu kỳ 3
for k in range(1, 13):
    s = k % n
    print(f"k={k:>2} gcd(6,k)={gcd(n, k)} bất biến? {row == row[s:] + row[:s]}")

False (kỳ vọng False )
True (kỳ vọng True )
True (kỳ vọng True )
k= 1 gcd(6,k)=1 bất biến? False
k= 2 gcd(6,k)=2 bất biến? False
k= 3 gcd(6,k)=3 bất biến? True
k= 4 gcd(6,k)=2 bất biến? False
k= 5 gcd(6,k)=1 bất biến? False
k= 6 gcd(6,k)=6 bất biến? True
k= 7 gcd(6,k)=1 bất biến? False
k= 8 gcd(6,k)=2 bất biến? False
k= 9 gcd(6,k)=3 bất biến? True
k=10 gcd(6,k)=2 bất biến? False
k=11 gcd(6,k)=1 bất biến? False
k=12 gcd(6,k)=6 bất biến? True


In [14]:
# Code nộp LeetCode cho Cách 3 (gọn, O(1) bộ nhớ phụ)
from typing import List

class Solution:
    def areSimilar(self, mat: List[List[int]], k: int) -> bool:
        n = len(mat[0])
        for row in mat:
            for j in range(n):
                if row[j] != row[(j + k) % n]:
                    return False
        return True

**Độ phức tạp Cách 4:** **Time `O(m · n + log n)`**, **Space `O(1)`**.

## 8. TRACE từng bước

Ta theo dõi (a) Cách 2 với từng hàng của Ví dụ 2; (b) Cách 3 so sánh `row[j]` với `row[(j + k) % n]` trên Ví dụ 1.

In [15]:
def trace_slice(mat, k):
    n = len(mat[0])
    s = k % n
    print(f"n={n} k={k} -> dịch s=k%n={s} ô")
    for i, row in enumerate(mat):
        if i % 2 == 0:
            shifted, name = row[s:] + row[:s], "trái "
        else:
            shifted, name = row[n - s:] + row[:n - s], "phải "
        same = shifted == row
        print(f"hàng {i} {name}: {row} -> {shifted} | giống gốc? {same}")
        if not same:
            print("=> False (dừng ngay)")
            return False
    print("=> True")
    return True


assert trace_slice([[1, 2, 1, 2], [5, 5, 5, 5], [6, 3, 6, 3]], 2) is True
assert trace_slice([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 4) is False

n=4 k=2 -> dịch s=k%n=2 ô
hàng 0 trái : [1, 2, 1, 2] -> [1, 2, 1, 2] | giống gốc? True
hàng 1 phải : [5, 5, 5, 5] -> [5, 5, 5, 5] | giống gốc? True
hàng 2 trái : [6, 3, 6, 3] -> [6, 3, 6, 3] | giống gốc? True
=> True
n=3 k=4 -> dịch s=k%n=1 ô
hàng 0 trái : [1, 2, 3] -> [2, 3, 1] | giống gốc? False
=> False (dừng ngay)


In [16]:
def trace_index(mat, k):
    n = len(mat[0])
    for i, row in enumerate(mat):
        for j in range(n):
            t = (j + k) % n
            ok = row[j] == row[t]
            print(f"hàng {i}: j={j} (j+k)%n={t} row[j]={row[j]} row[t]={row[t]} -> {'khớp' if ok else 'LỆCH'}")
            if not ok:
                print("=> False")
                return False
    print("=> True")
    return True


assert trace_index([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 4) is False
assert trace_index([[2, 2], [2, 2]], 3) is True

hàng 0: j=0 (j+k)%n=1 row[j]=1 row[t]=2 -> LỆCH
=> False
hàng 0: j=0 (j+k)%n=1 row[j]=2 row[t]=2 -> khớp
hàng 0: j=1 (j+k)%n=0 row[j]=2 row[t]=2 -> khớp
hàng 1: j=0 (j+k)%n=1 row[j]=2 row[t]=2 -> khớp
hàng 1: j=1 (j+k)%n=0 row[j]=2 row[t]=2 -> khớp
=> True


Bảng tóm tắt (Cách 3, Ví dụ 1, `n = 3`, `k = 4`, hàng 0 = `[1,2,3]`):

| `j` | `(j+k) % n` | `row[j]` | `row[(j+k)%n]` | kết quả |
|---|---|---|---|---|
| 0 | 1 | 1 | 2 | LỆCH, trả `False` |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (so sánh chỉ số, không xoay). Đúng class `Solution`, đúng method `areSimilar(self, mat: List[List[int]], k: int) -> bool`.

In [17]:
from typing import List

class Solution:
    def areSimilar(self, mat: List[List[int]], k: int) -> bool:
        n = len(mat[0])                                  # số cột
        for row in mat:
            for j in range(n):
                if row[j] != row[(j + k) % n]:           # ô j sau khi dịch nhận giá trị của ô (j+k)%n
                    return False
        return True


sol = Solution()
for mat, k, expected in examples:
    print(mat, k, "->", sol.areSimilar(mat, k))

[[1, 2, 3], [4, 5, 6], [7, 8, 9]] 4 -> False
[[1, 2, 1, 2], [5, 5, 5, 5], [6, 3, 6, 3]] 2 -> True
[[2, 2], [2, 2]] 3 -> True


## 10. Vì sao thuật toán đúng

- **Hàng độc lập.** Phép dịch tác động riêng từng hàng, nên ma trận giống ban đầu ⇔ **mọi** hàng giống ban đầu. Ta trả `False` ngay khi một hàng lệch và `True` nếu không hàng nào lệch.
- **Dịch trái `k` ô = đọc ô `(j + k) % n`.** Phần tử ở ô `j` sau khi dịch trái là phần tử vốn nằm cách nó `k` ô về bên phải, quấn vòng bằng `% n`. Hàng giống ban đầu ⇔ `row[j] == row[(j + k) % n]` với mọi `j`.
- **Dịch phải cũng vậy.** Dịch phải `k` ô: ô `j` nhận giá trị ô `(j - k) % n`. Đặt `t = (j - k) % n`, ta có `j = (t + k) % n`. Điều kiện `row[j] == row[(j - k) % n]` với mọi `j` đổi biến `j -> t` thành `row[(t + k) % n] == row[t]` với mọi `t`, **chính là điều kiện của dịch trái**. Vậy hai hướng cho cùng đáp án, và ta khỏi xét hàng chẵn/lẻ.
- **`k` lớn không là vấn đề.** `% n` quấn vòng nên mọi `k` đều đúng.

## 11. Phân tích Time / Space Complexity

Gọi `m` hàng, `n` cột, `k` số lần dịch.

- **Cách 1 (mô phỏng):** Time `O(k · m · n)`, Space `O(m · n)`.
- **Cách 2 (slicing một lần):** Time `O(m · n)`, Space `O(n)`.
- **Cách 3 (chỉ số `% n`):** Time `O(m · n)`, Space `O(1)`.
- **Cách 4 (`gcd`):** Time `O(m · n + log n)`, Space `O(1)`.
- Mọi cách phải đọc từng phần tử nếu đáp án là `True`, nên `O(m · n)` là tối ưu. Ở đây `m, n <= 25`, `k <= 50`: tối đa 625 ô.

## 12. Bộ test

In [18]:
sol = Solution()
impls = {
    "simulate": are_similar_simulate,
    "slice": are_similar_slice,
    "index": are_similar_index,
    "index_all": are_similar_index_all,
    "gcd": are_similar_gcd,
    "submit": sol.areSimilar,
}

for mat, k, expected in examples:
    for key, fn in impls.items():
        got = fn([r[:] for r in mat], k)
        assert got == expected, (key, mat, k, got, expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [19]:
def check_all(mat, k, expected, label):
    for key, fn in impls.items():
        got = fn([r[:] for r in mat], k)
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([[7]], 1, True, "ma trận 1x1")
check_all([[1, 2]], 1, False, "một hàng, n=2, k=1 (dịch trái đổi chỗ)")
check_all([[1, 2]], 2, True, "k = n: về chỗ cũ")
check_all([[1, 2, 3], [4, 5, 6]], 3, True, "k = n = 3: mọi hàng về chỗ cũ")
check_all([[1, 2, 3], [4, 5, 6]], 6, True, "k = 2n")
check_all([[1, 2, 3, 4]], 50, False, "k = 50, n = 4: 50 % 4 = 2, hàng đổi")
check_all([[1, 2, 1, 2]], 50, True, "k = 50 với chu kỳ 2: 50 % 2 = 0")
check_all([[1, 2, 1, 2], [3, 4, 5, 6]], 2, False, "hàng 0 giữ nguyên nhưng hàng 1 (hàng lẻ) lệch")
check_all([[1, 2, 3], [1, 1, 1], [3, 2, 1]], 1, False, "hàng 0 đã lệch")
check_all([[5] * 25 for _ in range(25)], 49, True, "25x25 toàn một giá trị")

PASS ma trận 1x1 -> True
PASS một hàng, n=2, k=1 (dịch trái đổi chỗ) -> False
PASS k = n: về chỗ cũ -> True
PASS k = n = 3: mọi hàng về chỗ cũ -> True
PASS k = 2n -> True
PASS k = 50, n = 4: 50 % 4 = 2, hàng đổi -> False
PASS k = 50 với chu kỳ 2: 50 % 2 = 0 -> True
PASS hàng 0 giữ nguyên nhưng hàng 1 (hàng lẻ) lệch -> False
PASS hàng 0 đã lệch -> False
PASS 25x25 toàn một giá trị -> True


### 12.2. Tính chất và phản ví dụ

In [20]:
# Tính chất 1: hướng dịch không quan trọng (mọi hàng dịch trái cho cùng đáp án)
def are_similar_all_left(mat, k):
    n = len(mat[0])
    s = k % n
    return all(row[s:] + row[:s] == row for row in mat)

def are_similar_all_right(mat, k):
    n = len(mat[0])
    s = k % n
    return all((row[n - s:] + row[:n - s] if s else row[:]) == row for row in mat)

random.seed(7)
for _ in range(500):
    m, n = random.randint(1, 6), random.randint(1, 6)
    period = random.choice([d for d in range(1, n + 1) if n % d == 0])
    mat = []
    for _ in range(m):
        base = [random.randint(1, 3) for _ in range(period)]
        mat.append((base * (n // period))[:n] if random.random() < 0.8 else [random.randint(1, 3) for _ in range(n)])
    k = random.randint(1, 50)
    assert are_similar_all_left(mat, k) == are_similar_all_right(mat, k) == are_similar_simulate(mat, k)
print("PASS dịch trái hay dịch phải cho cùng đáp án")

# Tính chất 2: k và k + n cho cùng đáp án
for _ in range(500):
    m, n = random.randint(1, 5), random.randint(1, 5)
    mat = [[random.randint(1, 2) for _ in range(n)] for _ in range(m)]
    k = random.randint(1, 20)
    assert sol.areSimilar(mat, k) == sol.areSimilar(mat, k + n)
print("PASS k và k + n cho cùng kết quả")

# Tính chất 3: đầu vào không bị sửa
mat = [[1, 2, 3], [4, 5, 6]]
copy = [r[:] for r in mat]
for key, fn in impls.items():
    fn(mat, 2)
assert mat == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: cắt lát không mod khi k > n
def sai_khong_mod(mat, k):
    return all(row[k:] + row[:k] == row for row in mat)

assert sai_khong_mod([[1, 2, 3]], 4) is True and sol.areSimilar([[1, 2, 3]], 4) is False
print("PASS phản ví dụ: row[k:] + row[:k] không quấn vòng khi k > n")

# Phản ví dụ 2: return True sớm
def sai_true_som(mat, k):
    n = len(mat[0])
    for row in mat:
        if all(row[j] == row[(j + k) % n] for j in range(n)):
            return True
    return False

assert sai_true_som([[1, 1], [1, 2]], 1) is True and sol.areSimilar([[1, 1], [1, 2]], 1) is False
print("PASS phản ví dụ: return True ngay ở hàng đầu bỏ qua hàng sau")

PASS dịch trái hay dịch phải cho cùng đáp án
PASS k và k + n cho cùng kết quả
PASS đầu vào không bị sửa
PASS phản ví dụ: row[k:] + row[:k] không quấn vòng khi k > n
PASS phản ví dụ: return True ngay ở hàng đầu bỏ qua hàng sau


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [21]:
random.seed(2946)
for trial in range(2000):
    m, n = random.randint(1, 25), random.randint(1, 25)
    k = random.randint(1, 50)
    divs = [d for d in range(1, n + 1) if n % d == 0]
    mat = []
    for _ in range(m):
        if random.random() < 0.85:
            p = random.choice(divs)
            base = [random.randint(1, 25) for _ in range(p)]
            mat.append(base * (n // p))              # hàng có chu kỳ p | n
        else:
            mat.append([random.randint(1, 25) for _ in range(n)])
    ref = are_similar_simulate([r[:] for r in mat], k)
    for key, fn in impls.items():
        assert fn([r[:] for r in mat], k) == ref, (key, mat, k)
print("PASS 2000 bộ ngẫu nhiên - mọi cách khớp nhau")

PASS 2000 bộ ngẫu nhiên - mọi cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên `k % n` khi cắt lát

In [22]:
def sai_khong_mod_2(mat, k):
    for i, row in enumerate(mat):
        if i % 2 == 0:
            shifted = row[k:] + row[:k]                   # k = 4, n = 3: row[4:] = [] nên shifted = row
        else:
            shifted = row[-k:] + row[:-k]
        if shifted != row:
            return False
    return True


print("sai  :", sai_khong_mod_2([[1, 2, 3], [4, 5, 6]], 4), "| đúng:", Solution().areSimilar([[1, 2, 3], [4, 5, 6]], 4))

sai  : True | đúng: False


**Vì sao sai:** cắt lát với `k >= n` **không quấn vòng**: `row[4:]` là rỗng và `row[:4]` là cả hàng, nên "kết quả dịch" bằng chính hàng ban đầu và ta kết luận `True` oan. Phải dùng `s = k % n` (hoặc chỉ số `% n`).

### Lỗi 2: Sửa luôn ma trận gốc rồi so sánh với chính nó

In [23]:
def sai_sua_goc(mat, k):
    cur = mat                                             # cur và mat là CÙNG một đối tượng
    for _ in range(k):
        for i in range(len(cur)):
            cur[i] = cur[i][1:] + cur[i][:1] if i % 2 == 0 else cur[i][-1:] + cur[i][:-1]
    return cur == mat                                     # luôn True


print("sai  :", sai_sua_goc([[1, 2, 3], [4, 5, 6]], 1), "| đúng:", Solution().areSimilar([[1, 2, 3], [4, 5, 6]], 1))

sai  : True | đúng: False


**Vì sao sai:** `cur = mat` không sao chép; mọi thay đổi trên `cur` là thay đổi trên `mat`, nên cuối cùng `cur == mat` luôn `True`. Phải sao chép từng hàng: `cur = [row[:] for row in mat]`.

### Lỗi 3: `return True` quá sớm

In [24]:
def sai_true_som_2(mat, k):
    n = len(mat[0])
    for row in mat:
        if all(row[j] == row[(j + k) % n] for j in range(n)):
            return True                                   # mới một hàng khớp đã kết luận
    return False


print("sai  :", sai_true_som_2([[1, 1], [1, 2]], 1), "| đúng:", Solution().areSimilar([[1, 1], [1, 2]], 1))

sai  : True | đúng: False


**Vì sao sai:** yêu cầu là **mọi** hàng phải giống ban đầu, nên phải trả `False` ngay khi **một** hàng lệch, và chỉ trả `True` sau khi đã duyệt hết. Dùng mẫu "tìm thấy phản ví dụ thì `False`, hết vòng lặp thì `True`".

### Lỗi 4: Nhầm hàng chẵn/lẻ (đếm từ 1)

In [25]:
def sai_chan_le(mat, k):
    n = len(mat[0])
    s = k % n
    for i, row in enumerate(mat):
        if i % 2 == 1:                                    # nhầm: hàng chỉ số 0 mới là hàng chẵn -> dịch trái
            shifted = row[s:] + row[:s]
        else:
            shifted = row[n - s:] + row[:n - s]
        if shifted != row:
            return False
    return True


# Hai cách nhầm hướng vẫn cho cùng đáp án (nhờ tính đối xứng), nhưng ta nên đọc đề chuẩn:
print(sai_chan_le([[1, 2, 3], [4, 5, 6]], 1), Solution().areSimilar([[1, 2, 3], [4, 5, 6]], 1))

False False


**Nhận xét:** lỗi này **vô hại với bài này** vì dịch trái/phải cho cùng đáp án (mục 10). Nhưng nó cho thấy cách kiểm tra: chỉ số hàng đầu tiên là `0` (chẵn). Đừng ỷ vào tính đối xứng trong những bài mà hướng thực sự quan trọng (ví dụ khi **in ra ma trận sau khi dịch**).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "dịch vòng / xoay vòng `k` lần" → nghĩ ngay `k % n` và công thức chỉ số `(j + k) % n`.
- Công thức nhớ: **"hàng bất biến dưới dịch `k` ⇔ `row[j] == row[(j + k) % n]` với mọi `j`"**.
- Dịch trái và dịch phải cùng `k` ô: một hàng **bất biến** dưới phép này ⇔ bất biến dưới phép kia.
- Cắt lát `a[p:] + a[:p]` chỉ đúng khi `0 <= p <= n`; nếu `p` có thể lớn, đưa về `p % n`.
- Muốn xoay thật sự nhiều lần hiệu quả: `collections.deque.rotate(-k)` (trái) / `rotate(k)` (phải), tự quấn vòng.
- Mẫu `for ... if sai: return False` rồi `return True` cho "mọi phần tử đều thỏa".

## 15. Biến thể của bài

1. **In ra ma trận** sau `k` lần dịch (lúc này hướng chẵn/lẻ quan trọng).
2. Tìm **`k` nhỏ nhất** (`k >= 1`) để ma trận giống ban đầu: đó là `lcm` của chu kỳ nhỏ nhất của từng hàng.
3. Dịch **cột** thay vì hàng (dịch lên/xuống): chuyển vị `zip(*mat)` rồi xử lý như hàng.
4. Mỗi hàng `i` dịch `i` ô (thay vì `k`): ma trận có giống ban đầu không?
5. Kiểm tra ma trận **giống sau dịch bất kỳ** (mọi `k`): mọi hàng phải gồm toàn giá trị bằng nhau.

Thử biến thể 1 và 2:

In [26]:
def shift_matrix(mat, k):
    n = len(mat[0])
    s = k % n
    out = []
    for i, row in enumerate(mat):
        d = deque(row)
        d.rotate(-s if i % 2 == 0 else s)            # rotate âm = trái, dương = phải
        out.append(list(d))
    return out


print("1.", shift_matrix([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 1))    # [[2,3,1],[6,4,5],[8,9,7]]
assert shift_matrix([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 1) == [[2, 3, 1], [6, 4, 5], [8, 9, 7]]


def smallest_period_of_row(row):
    n = len(row)
    return next(p for p in range(1, n + 1) if n % p == 0 and row == row[:p] * (n // p))


def smallest_k(mat):
    from math import lcm
    n = len(mat[0])
    return lcm(*[smallest_period_of_row(r) for r in mat])      # hàng bất biến dưới dịch p -> k phải là bội của p


mat = [[1, 2, 1, 2], [5, 5, 5, 5], [6, 3, 6, 3]]
k0 = smallest_k(mat)
print("2. k nhỏ nhất:", k0)
assert k0 == 2 and Solution().areSimilar(mat, k0)
assert all(not Solution().areSimilar(mat, k) for k in range(1, k0))

1. [[2, 3, 1], [6, 4, 5], [8, 9, 7]]
2. k nhỏ nhất: 2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng từng lần dịch | dịch 1 ô, lặp `k` lần | `O(k·m·n)` | `O(m·n)` | hiểu đúng đề; kiểm chứng |
| 2. Slicing một lần với `k % n` | `row[s:] + row[:s]` theo hướng đề | `O(m·n)` | `O(n)` | rõ theo đề, dùng khi cần ma trận sau dịch |
| 3. So sánh chỉ số `(j+k) % n` | không xoay, bỏ qua hướng | `O(m·n)` | `O(1)` | **cách nộp bài**: gọn và tiết kiệm bộ nhớ |
| 4. Chu kỳ `gcd(n, k)` | kiểm tra dịch `gcd` ô | `O(m·n)` | `O(1)` | hiểu cấu trúc toán học |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[1,2,3,4][1:] + [1,2,3,4][:1]`, `7 % 4`, `[5,6,7][-1:] + [5,6,7][:-1]`, `Solution().areSimilar([[3,3,3]], 5)`, `Solution().areSimilar([[1,2]], 1)`.

**Bài 2 (tính tay).** Với `mat = [[1,2,1,2],[3,4,3,4]]` hãy tính `areSimilar` cho `k = 1`, `k = 2`, `k = 3`, `k = 4`. Đoán quy luật theo chu kỳ của hàng.

**Bài 3 (điền TODO).** Hoàn thành `are_similar_todo(mat, k)` theo Cách 3.

In [27]:
def are_similar_todo(mat, k):
    n = len(mat[0])
    for row in mat:
        for j in range(n):
            # TODO 1: nếu row[j] khác row[(j + k) % n] thì trả về False
            pass
    # TODO 2: nếu không hàng nào lệch thì trả về True
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `are_similar_slice2(mat, k)` **theo đúng hướng đề** (hàng chẵn trái, hàng lẻ phải) bằng slicing với `s = k % n` mà không nhìn code mẫu. Kiểm tra ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `rotate_rows(mat, k)` trả ra ma trận sau khi dịch (hàng chẵn trái, hàng lẻ phải) bằng `deque`. (b) Viết `min_k(mat)` trả về `k >= 1` nhỏ nhất để ma trận giống ban đầu. (c) Viết `similar_for_all_k(mat)`: `True` nếu ma trận giống ban đầu với **mọi** `k`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[2,3,4,1]`, `3`, `[7,5,6]`, `True`, `False`. Bài 3: `if row[j] != row[(j + k) % n]: return False` rồi `return True` sau hai vòng lặp.
- **Mức 2 (vừa):** Bài 2: hàng `[1,2,1,2]` bất biến khi dịch số **chẵn**, hàng `[3,4,3,4]` cũng vậy. Nên: `k = 1` False, `k = 2` True, `k = 3` False, `k = 4` True. Bài 4: `s = k % n`, hàng chẵn `row[s:] + row[:s]`, hàng lẻ `row[n-s:] + row[:n-s]`.
- **Mức 3 (sâu):** Bài 5(a): `d = deque(row); d.rotate(-s)` là trái, `d.rotate(s)` là phải. 5(b): tìm chu kỳ nhỏ nhất `p | n` của mỗi hàng (`row == row[:p] * (n // p)`), rồi `k` nhỏ nhất là `lcm` các chu kỳ. 5(c): bất biến với `k = 1` ⇔ bất biến với mọi `k` ⇔ mọi hàng toàn giá trị bằng nhau (`len(set(row)) == 1`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
from math import lcm

# Bài 1
assert [1, 2, 3, 4][1:] + [1, 2, 3, 4][:1] == [2, 3, 4, 1] and 7 % 4 == 3
assert [5, 6, 7][-1:] + [5, 6, 7][:-1] == [7, 5, 6]
assert Solution().areSimilar([[3, 3, 3]], 5) is True
assert Solution().areSimilar([[1, 2]], 1) is False

# Bài 2
mat2 = [[1, 2, 1, 2], [3, 4, 3, 4]]
assert [Solution().areSimilar(mat2, k) for k in (1, 2, 3, 4)] == [False, True, False, True]

# Bài 3
def are_similar_todo_ans(mat, k):
    n = len(mat[0])
    for row in mat:
        for j in range(n):
            if row[j] != row[(j + k) % n]:
                return False
    return True

for mat, k, expected in examples:
    assert are_similar_todo_ans(mat, k) == expected

# Bài 4
def are_similar_slice2(mat, k):
    n = len(mat[0])
    s = k % n
    for i, row in enumerate(mat):
        shifted = row[s:] + row[:s] if i % 2 == 0 else row[n - s:] + row[:n - s]
        if shifted != row:
            return False
    return True

for mat, k, expected in examples:
    assert are_similar_slice2(mat, k) == expected

# Bài 5(a)
def rotate_rows(mat, k):
    s = k % len(mat[0])
    out = []
    for i, row in enumerate(mat):
        d = deque(row)
        d.rotate(-s if i % 2 == 0 else s)
        out.append(list(d))
    return out

assert rotate_rows([[1, 2, 3], [4, 5, 6]], 4) == [[2, 3, 1], [6, 4, 5]]

# Bài 5(b)
def min_k(mat):
    n = len(mat[0])
    periods = [next(p for p in range(1, n + 1) if n % p == 0 and r == r[:p] * (n // p)) for r in mat]
    return lcm(*periods)

assert min_k([[1, 2, 1, 2], [5, 5, 5, 5]]) == 2
assert min_k([[1, 2, 3], [4, 5, 6]]) == 3
assert min_k([[1, 2, 1, 2, 1, 2], [1, 2, 3, 1, 2, 3]]) == 6
for mat in ([[1, 2, 1, 2], [5, 5, 5, 5]], [[1, 2, 3], [4, 5, 6]]):
    k0 = min_k(mat)
    assert Solution().areSimilar(mat, k0) and not any(Solution().areSimilar(mat, k) for k in range(1, k0))

# Bài 5(c)
def similar_for_all_k(mat):
    return all(len(set(row)) == 1 for row in mat)

assert similar_for_all_k([[2, 2], [3, 3, 3][:2]]) is True
assert similar_for_all_k([[1, 2], [3, 3]]) is False
assert all(Solution().areSimilar([[2, 2, 2], [3, 3, 3]], k) for k in range(1, 51))
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hàng bất biến dưới dịch `k` ⇔ `row[j] == row[(j + k) % n]`)

```python
from typing import List

class Solution:
    def areSimilar(self, mat: List[List[int]], k: int) -> bool:
        n = len(mat[0])
        for row in mat:
            for j in range(n):
                if row[j] != row[(j + k) % n]:
                    return False
        return True
```

**Checklist**

- [ ] Đã dùng `% n` (hoặc `k % n`) để `k > n` vẫn đúng?
- [ ] Nếu cắt lát: đã đổi `k` thành `s = k % n` trước khi `row[s:] + row[:s]`?
- [ ] Nếu mô phỏng: đã **sao chép từng hàng** (`[r[:] for r in mat]`), không sửa ma trận gốc?
- [ ] Trả `False` ngay khi **một** hàng lệch, trả `True` chỉ sau khi duyệt hết?
- [ ] Hiểu vì sao hướng trái/phải không đổi đáp án (hoặc đã xử lý đúng hướng khi cần in ma trận)?
- [ ] Đã thử: ma trận `1x1`, `k = n`, `k = 50`, hàng có chu kỳ nhỏ, hàng toàn giá trị bằng nhau?
- [ ] Skeleton đúng `class Solution` / `def areSimilar(self, mat: List[List[int]], k: int) -> bool`?